# 11.4 Reading Execution Plans

## Learning Objectives

By the end of this lesson you will be able to:

- Understand what an Execution Plan is
- Learn how to use explain()
- Understand Logical Plans
- Understand Optimized Logical Plans
- Understand Physical Plans
- Learn how Spark decides execution strategies

> **Core Rule:** Execution Plans help Data Engineers understand what Spark is actually doing behind the scenes.

## Setup: Initialize Spark

Let's start our local Spark Session and generate some mock sales data so we can generate real execution plans in this notebook.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("Reading_Execution_Plans") \
    .master("local[*]") \
    .getOrCreate()

# Create mock sales data (replacing sales.csv so this notebook runs independently)
data = [
    (1, 101, "New York", 1500.0, "2023-01-01"),
    (2, 102, "London", 800.0, "2023-01-02"),
    (3, 101, "New York", 2000.0, "2023-01-03"),
    (4, 103, "Paris", 500.0, "2023-01-04"),
    (5, 102, "London", 1200.0, "2023-01-05")
]
columns = ["order_id", "customer_id", "city", "amount", "date"]

df = spark.createDataFrame(data, columns)
print("Spark Session Initialized and Mock Data Created!")

# Why Learn Execution Plans?

Many beginners write Spark code and only focus on the output.

However, in production environments, engineers often ask:

- Why is this query slow?
- Why is Spark performing a Shuffle?
- Why is a Join taking so long?

To answer these questions, we need to understand Spark's Execution Plan.

Execution Plans reveal how Spark intends to process your data.

# What is an Execution Plan?

An Execution Plan is Spark's roadmap for executing a query.

Before Spark starts processing data, it creates a plan describing:

- What operations are required
- In what order operations should execute
- How data should move through the cluster

Think of it as a GPS route before starting a journey.

# Example Query

Suppose we write:

1. Read sales data
2. Filter orders above ₹1000
3. Group by customer
4. Calculate total sales

The code looks simple.

However, Spark internally creates multiple execution plans before running the query.

<h3>Query Planning Process</h3>

<img src="../assets/Module_11/11_04_01.png" alt="Query Planning Workflow" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark query planning workflow. User writes DataFrame code leading to Logical Plan, Optimized Logical Plan, Physical Plan, and finally Execution. Educational architecture diagram with arrows. White background.</i></p>

# The Catalyst Optimizer

Spark uses a component called the Catalyst Optimizer.

Catalyst is responsible for:

- Analyzing queries
- Optimizing queries
- Generating execution plans

Whenever you write DataFrame or Spark SQL code, Catalyst begins working automatically.

# The Three Major Plans

Spark generally creates:

1. Logical Plan
2. Optimized Logical Plan
3. Physical Plan

Each stage improves the execution strategy before actual processing begins.

In [ ]:
# Let's write the query we described earlier.
# Note: Because of Lazy Evaluation, this query is NOT executed yet.
# Spark is just building the Execution Plan.

result = (
    df.filter("amount > 1000")
      .groupBy("customer_id")
      .sum("amount")
)

print("Query result:")
result.show()

# Viewing the Execution Plan

Spark provides the `explain()` method.

This method displays the execution plans generated by Spark.

It is one of the most useful debugging tools for Data Engineers.

In [ ]:
# By default, explain() shows the final Physical Plan
print("=== DEFAULT EXPLAIN (Physical Plan) ===")
result.explain()

# What Does explain() Show?

Depending on Spark version and configuration, `explain()` may display:

- Parsed Logical Plan
- Analyzed Logical Plan
- Optimized Logical Plan
- Physical Plan

For beginners, focus primarily on:

- Logical Plan
- Optimized Logical Plan
- Physical Plan

*(Note: You can pass `extended=True` to see all plans: `result.explain(True)`)*

<h3>Output of explain()</h3>

<img src="../assets/Module_11/11_04_02.png" alt="Output of explain" width="75%">

<p><i><b>Image Prompt:</b> Spark explain() output showing Parsed Logical Plan, Optimized Logical Plan and Physical Plan sections. Educational screenshot-style visualization. Clean white background.</i></p>

# Logical Plan

The Logical Plan represents:

"What the user wants."

At this stage Spark focuses on the requested operations rather than execution details.

Examples:
- Read data
- Filter rows
- Group records
- Aggregate values

No optimization has occurred yet.

# Example Logical Plan

Read Sales Data
    ↓
Filter Amount > 1000
    ↓
Group By Customer
    ↓
Calculate Sum

This plan describes business logic only.

It does not describe how Spark will execute the query.

<h3>Logical Plan</h3>

<img src="../assets/Module_11/11_04_03.png" alt="Logical Plan Flowchart" width="75%">

<p><i><b>Image Prompt:</b> Spark Logical Plan diagram showing Scan → Filter → GroupBy → Aggregate. Simple flowchart focused on business logic rather than execution details.</i></p>

# Why Logical Plans Matter

Logical Plans help engineers understand:

- Requested operations
- Query structure
- Business requirements

They are useful for verifying that Spark correctly interpreted your code.

# Optimized Logical Plan

After generating the Logical Plan, Spark begins optimization.

Catalyst applies optimization rules.

The result is called the Optimized Logical Plan.

This version is usually smaller, faster, and more efficient.

# Common Optimizations

Catalyst may perform:

- Predicate Pushdown
- Column Pruning
- Constant Folding
- Filter Reordering
- Expression Simplification

These optimizations reduce unnecessary work.

# Predicate Pushdown Example

Suppose we only need records where:
`amount > 1000`

Instead of reading all records first and filtering later, Spark attempts to push the filter closer to the data source (like reading from a Parquet file).

This reduces data movement and improves performance.

<h3>Predicate Pushdown</h3>

<img src="../assets/Module_11/11_04_04.png" alt="Predicate Pushdown" width="75%">

<p><i><b>Image Prompt:</b> Before and after Predicate Pushdown illustration. Before: Read entire dataset then filter. After: Filter applied during data read. Educational Spark optimization infographic.</i></p>

# Column Pruning Example

Suppose a table contains:
- customer_id
- name
- city
- amount
- date

But we only need:
- customer_id
- amount

Spark may read only the required columns. This optimization is called Column Pruning.

Let's simulate both Pushdown and Pruning using Parquet files.

In [ ]:
import shutil
out_path = "/tmp/sales_parquet"

# 1. Save data as Parquet (a columnar storage format that supports Pushdown & Pruning)
df.write.mode("overwrite").parquet(out_path)

# 2. Read it back
df_file = spark.read.parquet(out_path)

# 3. Apply filter and select (Triggering Pushdown & Pruning)
df_optimized = df_file.filter(F.col("amount") > 1000).select("customer_id", "amount")

print("Notice in the Physical Plan below:")
print("1. Look for 'PushedFilters: [IsNotNull(amount), GreaterThan(amount,1000.0)]'")
print("2. Look for 'ReadSchema: struct<customer_id:bigint,amount:double>'")
print("-"*60)
df_optimized.explain()
print("-"*60)

# Cleanup
shutil.rmtree(out_path, ignore_errors=True)

<h3>Column Pruning</h3>

<img src="../assets/Module_11/11_04_05.png" alt="Column Pruning" width="75%">

<p><i><b>Image Prompt:</b> Spark Column Pruning visualization. Large table with many columns reduced to only required columns before processing. Beginner-friendly optimization diagram.</i></p>

# Optimized Logical Plan Example

**Original:**
Read
→ Filter
→ Select
→ Aggregate

**After Optimization:**
Read Only Required Columns
→ Apply Filter Early
→ Aggregate

Same result.
Less work.

<h3>Optimized Logical Plan</h3>

<img src="../assets/Module_11/11_04_06.png" alt="Optimized Logical Plan" width="75%">

<p><i><b>Image Prompt:</b> Side-by-side comparison showing Logical Plan and Optimized Logical Plan. Optimized version contains fewer operations and reduced data movement. Educational infographic.</i></p>

# Physical Plan

The Physical Plan answers:

"How will Spark execute the query?"

At this stage Spark selects actual execution strategies.

Examples:
- Broadcast Join
- Sort-Merge Join
- Shuffle Hash Join

This is the plan that Spark eventually executes.

# Physical Plan Contains

Physical Plans may include:

- Scan Operators
- Filter Operators
- Exchange Operators
- Join Operators
- Aggregation Operators

These operators directly impact performance.

<h3>Physical Plan</h3>

<img src="../assets/Module_11/11_04_07.png" alt="Physical Plan Tree" width="75%">

<p><i><b>Image Prompt:</b> Spark Physical Plan tree showing Scan, Filter, Exchange, Join and Aggregate operators. Technical but beginner-friendly execution diagram.</i></p>

# Exchange Operator

One operator deserves special attention:

**Exchange**

Whenever you see Exchange in a Physical Plan, it often indicates:

- Data movement
- Partition redistribution
- Shuffle activity

Exchange operators frequently contribute to performance costs. Let's force an Exchange by joining two tables.

In [ ]:
# Create two large dataframes
df_a = spark.range(1, 10000).withColumnRenamed("id", "user_id")
df_b = spark.range(1, 10000).withColumn("user_id", F.col("id") * 2)

# Perform a Join (Disabling Broadcast to force a Shuffle/Exchange)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

join_result = df_a.join(df_b, "user_id")

print("Look for the 'Exchange' operator indicating a Shuffle:")
print("-"*40)
join_result.explain()
print("-"*40)

# Reset config
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

<h3>Exchange Operator and Shuffle</h3>

<img src="../assets/Module_11/11_04_08.png" alt="Exchange Operator" width="75%">

<p><i><b>Image Prompt:</b> Spark Physical Plan highlighting Exchange operator causing shuffle between executors. Arrows showing data redistribution across cluster nodes.</i></p>

# Why Physical Plans Matter

Physical Plans help engineers answer:

- Which join strategy was selected?
- Is Spark performing a Shuffle?
- Is a Broadcast Join being used?
- Where is performance being lost?

Most production optimization starts here.

# Formatted Execution Plans

Formatted plans are easier to read.

They display:
- Operator hierarchy
- Metrics
- Data flow

Many engineers prefer formatted plans during debugging. Let's look at the formatted plan for our join.

In [ ]:
join_result.explain(mode="formatted")

<h3>Formatted Physical Plan</h3>

<img src="../assets/Module_11/11_04_09.png" alt="Formatted Execution Plan" width="75%">

<p><i><b>Image Prompt:</b> Modern formatted Spark execution plan displaying operator hierarchy, execution tree and metrics. Educational dashboard-style visualization.</i></p>

# Real-World Investigation Example

Suppose a query is running slowly.

You execute:
`result.explain()`

The Physical Plan reveals:

SortMergeJoin
    ↓
Exchange
    ↓
Exchange

Now you know:
- Spark is performing Shuffle
- Sort-Merge Join is being used
- Join optimization may improve performance

Execution Plans reveal the root cause.

# Key Takeaways

- Execution Plans show how Spark processes data.
- `explain()` displays query plans.
- Logical Plan represents business logic.
- Optimized Logical Plan applies Catalyst optimizations.
- Physical Plan contains actual execution operators.
- Exchange operators often indicate Shuffle.
- Understanding execution plans is a critical Spark optimization skill.

# Next Lesson

## 11.5 Join Strategies Overview

In the next lesson we will learn:
- Broadcast Join
- Sort Merge Join
- Shuffle Hash Join

Broadcast Joins are among the most effective Spark optimization techniques.

---

## Lesson Statistics

* **Topic:** Reading Execution Plans
* **Cells:** 36
* **Images:** 9
* **Estimated Duration:** 10–12 Minutes
* **Difficulty:** Beginner-Friendly
* **Foundation For:** Broadcast Joins, Sort-Merge Joins, AQE, Query Optimization, Spark Performance Tuning